# Webhooks & Event-Driven Patterns

In [ ]:
# Day 52 — Webhooks & Event-Driven Patterns
# Cron jobs (Day 51) run on a fixed schedule (e.g. every night at 2 AM).
# Webhooks react INSTANTLY when something happens:
#   - New data pushed to GitHub/DVC -> trigger ML retraining
#   - Model promoted to 'Production' in MLflow -> auto-deploy to K8s
#   - Prometheus Alertmanager fires -> trigger AI SRE diagnosis
#
# Today we'll cover:
#   1. Building a FastAPI webhook receiver
#   2. Verifying webhook signatures with `hmac` (security!)
#   3. Running heavy work in FastAPI `BackgroundTasks`
#   4. Idempotency & triggering Prefect/Airflow flows

### 1. FastAPI Webhook Receiver with HMAC Verification

In [ ]:
# NEVER trust an unauthenticated webhook endpoint on the internet!
# GitHub/GitLab sign every payload with a shared secret using HMAC-SHA256.

import os
os.makedirs("webhook_demo", exist_ok=True)

webhook_code = '''import hmac
import hashlib
import os
from fastapi import FastAPI, Request, Header, HTTPException, BackgroundTasks

app = FastAPI(title="MLOps Webhook Listener")
WEBHOOK_SECRET = os.getenv("WEBHOOK_SECRET", "dev-secret").encode()

def verify_signature(payload: bytes, signature_header: str) -> bool:
    if not signature_header or not signature_header.startswith("sha256="):
        return False
    expected = "sha256=" + hmac.new(WEBHOOK_SECRET, payload, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, signature_header)

def trigger_retraining(repo: str, commit: str):
    print(f"[BACKGROUND] Triggering ML retraining for {repo} @ {commit}")

@app.post("/webhooks/github")
async def github_webhook(
    request: Request,
    background_tasks: BackgroundTasks,
    x_hub_signature_256: str = Header(default="")
):
    body = await request.body()
    if not verify_signature(body, x_hub_signature_256):
        raise HTTPException(status_code=401, detail="Invalid webhook signature")

    data = await request.json()
    repo = data.get("repository", {}).get("full_name", "unknown")
    commit = data.get("after", "unknown")[:7]

    # Return 202 quickly; run heavy pipeline in background!
    background_tasks.add_task(trigger_retraining, repo, commit)
    return {"status": "accepted", "repo": repo, "commit": commit}
'''

with open("webhook_demo/app_webhook.py", "w") as f:
    f.write(webhook_code)

print("Created webhook_demo/app_webhook.py")

### 2. Testing HMAC Signature Generation Locally

In [ ]:
import hmac, hashlib, json

secret = b"dev-secret"
payload = json.dumps({"repository": {"full_name": "org/ml-data"}, "after": "abc1234567"}).encode()
sig = "sha256=" + hmac.new(secret, payload, hashlib.sha256).hexdigest()

print("Generated Signature Header:", sig)
print("Use `hmac.compare_digest()` to prevent timing attacks!")

In [ ]:
# Practice Questions:
#
# 1. Why should a webhook endpoint return HTTP 200/202 immediately and use BackgroundTasks?
# 2. Why do we use `hmac.compare_digest()` instead of `==` for signature checks?
# 3. What is idempotency and why does it matter if GitHub sends the same webhook twice?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: GitHub Push Webhook Receiver
# Filter push events so retraining ONLY triggers when files inside `data/` changed.

# Exercise 2: Idempotent Event Deduplicator
# Track processed `X-GitHub-Delivery` UUIDs in a set/dict so duplicate deliveries
# are safely ignored.

# Exercise 3: Alertmanager -> Slack/Teams Notifier
# Build a `/webhooks/alertmanager` POST endpoint that parses Prometheus alert JSON
# and formats a clean summary message.